## All about Data Quality and Non-Collision Backgrounds

<CENTER><img src="../../images/ATLASOD.gif" style="width:50%"></CENTER>

This notebook uses [ATLAS Open Data](http://opendata.atlas.cern) [2025 beta release](https://opendata.atlas.cern/docs/data/for_education/13TeV25_details) to teach you a bit about the concepts of "Data Quality" and "Non-Collision Backgrounds". It is intended for an education audience and is written to be accessible to a wide range of students.

### What is Data Quality?

Data quality is one of the most important concepts in any data analysis. Imagine asking 10 friends to measure the size of a box. They should all get about the same numbers. They might not get exactly the same numbers, though, and you might want to average all of the measurements together. If it turns out that two of your friends were trying to measure the box during a serious earthquake, then it might be reasonable to ignore their measurements and consider only the other eight. That's a simple example of a measurement rejection for data quality reasons. These sorts of issues come up all the time in real-life data analyses: a single weather station might be broken, and so its data is ignored when producing a weather report or forecast, for example.

For a variety of reasons, not every event recorded with the ATLAS detector is used in our data analysis. When the experiment is running, sometimes things go wrong. These can result in bad data quality, and events that have to be set aside. For example, some parts of the ATLAS detector are only turned on after collisions are taking place in the LHC -- they are disabled until the proton beams in the LHC have been ramped up to full energy and are completely stable, for safety reasons. Any events recorded before these parts of the detector are completely on and ready to provide good data are marked "bad".

Within the ATLAS data, there are special flags used by some detector systems to indicate that an individual event should not be used in a data analysis. The collaboration also produces "Good Runs Lists", which are lists of all the blocks of data that *should* be included in an analysis.

Of course, different analyses are affected by different issues. If you want to measure how loud a car stereo is, you might not care if some of the tires don't have enough air in them. Our collaboration tries to balance the possible number of combinations and the importance of changes: we have only a few good runs lists that are sufficient for most data analyses.

### What are Non-collision Backgrounds?

ATLAS is built to study collisions between protons (and between heavy ions) that happen right in the center of the detector. It does that by carefully measuring all the particles that come flying out of each collision, like a camera. That also leaves it sensitive to a variety of other things that can happen, which are generally described as "Non-collision backgrounds". The most common examples of non-collision backgrounds are:

* [Cosmic rays](https://en.wikipedia.org/wiki/Cosmic_ray). These are usually muons that come from the upper atmosphere and fly through the detector (they also fly through *you*, as you're sitting there, all the time). They look just like muons that come from collisions, except they usually come at funny angles (because they aren't passing straight through the center of the detector), and they often come at strange times (they can arrive when the collider has no protons in it! They can also come *between* bunches of protons).

* Detector noise. Just like static on a radio, the electronics in our detector sometimes have noise. Usually these look different from the signals we are looking for (just like you can tell the difference between static on your radio and music). Sometimes the static is so loud we can't hear the music any more, and then the data have to be discarded.

* Beam halo. As the protons fly around the ring of the LHC in bunches, there are [collimators](https://en.wikipedia.org/wiki/Collimator) that block any protons that get too far out of a normal orbit. Those protons don't just stop; they can produce sprays of particles that run parallel to the original proton beam and sometimes find their way into the detector. They usually come at funny angles, and move across the detector from one side to the other, rather than coming out of the center like particles from a collision.

* Beam gas. The proton bunches go around the LHC in a beam pipe that is *nearly* a vacuum, but it's not perfect. Sometimes there can be collisions between protons from the beam and stray atoms of gas in the beam pipe. They tend to be very asymmetric-looking (because the gas is drifting slowly, and the beam is running full-speed into it), and they tend to be off-center (because the gas can be anywhere in the beam pipe, not only in the very center).

One reason that data quality and non-collision backgrounds are intricately linked is that sometimes there is a period of time that suffers from serious non-collision backgrounds, and therefore must be excluded from our good runs lists.

## Getting into the data

Now that we know the ideas, we're going to get into using the ATLAS Open Data to look at non-collision backgrounds and data quality!

Because these backgrounds and data quality issues are cut out of the data, we don't try to model them in our [detector simulation](https://opendata.atlas.cern/docs/documentation/monte_carlo/introduction_MC). That means we are going to look only into the data for this analysis.

## ATLAS Open Data Initialisation

### First time package installation on your computer (not needed on mybinder)
This first cell installs the required python packages.
It only needs to be run the first time you open this notebook on your computer. 
If you close Jupyter and re-open on the same computer, you won't need to run this first cell again.

If this is opened on mybinder, you don't need to run this cell.

In [1]:
import sys
import os.path
!pip install atlasopenmagic
from atlasopenmagic import install_from_environment
install_from_environment()

Installing packages: ['aiohttp>=3.9.5', 'atlasopenmagic>=1.0.1', 'awkward>=2.6.7', 'awkward-pandas>=2023.8.0', 'coffea~=0.7.0', 'hist>=2.8.0', 'ipykernel>=6.29.5', 'jupyter>=1.0.0', 'lmfit>=1.3.2', 'matplotlib>=3.9.1', 'metakernel>=0.30.2', 'notebook<7', 'numpy>=1.26.4', 'pandas>=2.2.2', 'papermill>=2.6.0', 'pip>=24.2', 'scikit-learn>=1.5.1', 'uproot>=5.3.10', 'uproot3>=3.14.4', 'fsspec-xrootd>=0.5.1', 'jupyterlab_latex~=3.1.0', 'vector>=1.4.1']
  Using cached coffea-0.7.29-py2.py3-none-any.whl.metadata (9.6 kB)
  Using cached numpy-2.3.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (62 kB)
INFO: pip is looking at multiple versions of coffea to determine which version is compatible with other requirements. This could take a while.
  Using cached coffea-0.7.28-py2.py3-none-any.whl.metadata (9.6 kB)
  Using cached coffea-0.7.27-py2.py3-none-any.whl.metadata (9.6 kB)
  Using cached coffea-0.7.26-py2.py3-none-any.whl.metadata (9.6 kB)
  Using cached coffea-0.7.25-py2.py3-none-any.whl.met

We're going to import a number of packages to help us:
* `numpy`: provides numerical calculations such as histogramming
* `matplotlib`: common tool for making plots, figures, images, visualisations
* `uproot`: processes `.root` files typically used in particle physics into data formats used in python
* `awkward`: introduces `awkward` arrays, a format that generalizes `numpy` to nested data with possibly variable length lists
* `vector`: to allow vectorized 4-momentum calculations

In case this fails the first time you run it, and you've just installed new packages, you might just need to restart the jupyter kernel and try again.

In [2]:
import numpy as np # for numerical calculations such as histogramming
import matplotlib.pyplot as plt # for plotting
import matplotlib_inline # to edit the inline plot format
#matplotlib_inline.backend_inline.set_matplotlib_formats('pdf', 'svg') # to make plots in pdf (vector) format
from matplotlib.ticker import AutoMinorLocator # for minor ticks
import uproot # for reading .root files
import awkward as ak # to represent nested data in columnar format
import hist # for histogramming

We will use the [atlasopenmagic](https://opendata.atlas.cern/docs/data/atlasopenmagic) to access the open data directly from the ATLAS OpenData Portal so no need to download any samples. First we need to set the right release.

In [3]:
import atlasopenmagic as atom
atom.available_releases()
atom.set_release('2025e-13tev-beta')

Available releases:
2016e-8tev        2016 Open Data for education release of 8 TeV proton-proton collisions (https://opendata.cern.ch/record/3860).
2020e-13tev       2020 Open Data for education release of 13 TeV proton-proton collisions (https://cern.ch/2r7xt).
2024r-pp          2024 Open Data for research release for proton-proton collisions (https://opendata.cern.record/80020).
2024r-hi          2024 Open Data for research release for heavy-ion collisions (https://opendata.cern.ch/record/80035).
2025e-13tev-beta  2025 Open Data for education and outreach beta release for 13 TeV proton-proton collisions (https://opendata.cern.ch/record/93910).
2025r-evgen       2025 Open Data for research release for event generation (https://opendata.cern.ch/record/160000).
Active release set to: 2025e-13tev-beta. Metadata cache cleared.


## Example 1: Bad jets 

The Monte Carlo simulation is free from non-collision backgrounds and data quality issues, so we will look directly at the detector data for examples.

In [4]:
# Build a list of the data samples to go over; we'll just get the locations of the files we need
data_samples = atom.get_urls('data', protocol='https')

Fetching and caching all metadata for release: 2025e-13tev-beta...
Successfully cached 374 datasets.


One give-away that something has gone wrong in an event (that a non-collision background or some other data quality issue is present) is a large amount of missing transverse momentum. In the LHC, the proton beams collide head-on. Because of momentum conservation, if there are particles that fly out in one direction, some particles have to fly out in the opposite direction. An imbalance in the momentum in the collision might be an indication of detector signals that aren't coming from a collision. Let's select events with no leptons (for the more advanced student: this will help remove Standard Model events that often have missing transverse momentum from neutrinos, like the decays of W bosons into a lepton and a neutrino), at least one jet above 200 GeV, and at least 100 GeV of missing transverse momentum.

In [5]:
# Define what variables are important to our analysis
# We want the number of leptons, number of jets, the transverse momenta and phi angles of the jets, and the amount of missing transverse momentum
variables = ['lep_n','jet_n','jet_pt','jet_phi','met']

Now we can run over all the data and see what events pass our selections

In [ ]:
# We'll use the hist package to define a histogram in advance
# We'll be filling it as we go along, so that we save minimal data in memory
phihist = hist.Hist(hist.axis.Regular(40, -np.pi, np.pi, label="Leading jet phi"))

for afile in data_samples: 
    # Tell the folks how we are doing
    print(f'Working on file {afile}')
    tree = uproot.open('simplecache::'+ afile + ":analysis")

    # Perform the cuts for each data entry in the tree
    # the data will be in the form of an awkward array because we are using the library 'ak' (for awkward)
    for n,data in enumerate(tree.iterate(variables, library="ak")):
        # Let's let folks know how things are going regularly, so they don't get worried
        if (n+1)%10==0:
            print(f'Working on data chunk {n+1}')

        # In this array programming setup, cuts are defined like boolean objects
        # They can be added together with &

        # Start with a cut: Require zero leptons in the event
        cut = (data['lep_n']==0)

        # Add a cut: Require at least one jet
        cut = cut & (data['jet_n']>0)

        # Add a cut: Require at least 100 GeV of MET
        cut = cut & (data['met']>100)

        # Add a cut: Require at least one jet with at least 200 GeV of pT
        cut = cut & (data['jet_pt']>150)

        # Plot the phi of the highest pT jet after the above cuts are applied
        # This is tricky syntax! We get the 'jet_phi' values from data;
        # We apply the cut, and we ask for the first element of the array
        # (because the jet variables are sorted by the pT of the jet, this
        # gives us the phi of the highest transverse momentum jet)
        # Then we flatten the data with awkward - that strips out all the
        # events that didn't pass our cuts, and gives us just a list of the
        # phi values for the jets. Finally, we fill the histogram with that
        # list of phi values
        phihist.fill( ak.flatten(data['jet_phi'][cut][:,:1]) )

Working on file https://opendata.cern.ch/eos/opendata/atlas/rucio/user/egramsta/data15_periodD.noskim.root
Working on data chunk 10
Working on file https://opendata.cern.ch/eos/opendata/atlas/rucio/user/egramsta/data15_periodE.noskim.root


We can now plot the data using Matplotlib. Most of the code here is for the aesthetics of the plot.

In [ ]:
# We're going to make a plot with the matplotlib package
# It's quite powerful and has lots of nice features, but the syntax takes some getting used to

# Set up the figure and the axes on which we'll draw
fig, ax = plt.subplots(figsize=(8, 5))

# Convert from our hist histogram to a numpy array of values
y,bin_edges = phihist.to_numpy()
# Calculate errors
# The statistical uncertainty is just the square root of the number of counts
yerr = np.sqrt(y)
# Slightly awkward - convert from bin edges, which is what hist gives us, into bin centers, which is what matplotlib wants
x = [ (bin_edges[n]+bin_edges[n+1])/2. for n in range(len(bin_edges)-1) ]

# Now we can define the actual plot
ax.errorbar(x=x, y=y, yerr=yerr,
                    fmt='ko', # 'k' means black and 'o' is for circles 
                    label='Data') 

# x-axis label
ax.set_xlabel(r'Leading jet phi',
                    fontsize=13, x=1, horizontalalignment='right')

# write y-axis label for main axes
ax.set_ylabel('Jets',
                     fontsize=13, y=1, horizontalalignment='right') 

# set y-axis limits for main axes
ax.set_ylim( bottom=0, top=np.amax(y)*1.2 )

# add minor ticks on y-axis for main axes
ax.yaxis.set_minor_locator( AutoMinorLocator() ) 

# draw the legend
ax.legend( frameon=False ); # no box around the legend

A great success! Now, what does the plot mean?

What we're showing here is the angle of the highest transverse momentum jet in the event. The detector is designed to be symmetrical, like a can. The proton collisions don't know which way is up. So, if there are no detector issues and no non-collision backgrounds, this should be a perfectly flat distribution. It's not!

There are some small bumps that you can see around -2.5 to -0.5, and 0.5 to 2.5. Those are generally because some parts of the detector had problems. In this case, they weren't severe enough problems that we removed the data, but if a particular data analysis is very sensitive, it might have to apply corrections to fix those issues.

You can see two *big* excesses at 0 and pi (the edges of the histogram). Those are because of non-collision backgrounds! The way the collimators are arranged, the shielding of the detector, and even the physical structures supporting it (which are often made of steel) block more of the non-collision background at some angles than at others, and those bumps show areas where there's additional background created that doesn't get blocked.

For some data analyses, this is very dangerous! For example, if someone is searching for new physics that leaves a signal in the detector like a high-momentum jet and missing transverse momentum (that's what dark matter would look like in our detector!), then they need to be able to cut down on the amount of non-collision background. We have various tricks to do that (some of which are called "jet cleaning", because they are built to identify jets that have particularly funny properties in the detector) that the most sensitive analyses have to carefully apply.

Feel free to play around with the jet transverse momentum and missing transverse momentum cuts to see how you can enhance the non-collision background in the plot. If you don't require any missing transverse momentum, the background is totally invisible!

## Example 2: Reading Monte-Carlo data


Using the Standard Model, 
    we can do a set of randomised simulations to produce a set of theoretical data points to compare to our ATLAS data.
These are known as Monte-Carlo(MC) simulations.
There is one important change to be made to the MC data before we can compare them with our ATLAS data:
 - **Weights** - The MC data was computed in ideal circumstances. The real ATLAS detector has some inefficiencies, which we can account for by attributing the appropriate weight to each data point. The weight of a data point affects how it contributes to the histogram count for its bin.

Let's open an MC file.

In [ ]:
# We open an MC data file with sample value "Zee" using samples and infofile for reference of filenames
value = samples[r'Background $Z,t\bar{t},t\bar{t}+V,VVV$']["list"][0]

# This is now appended to our file path to retrieve the root file
background_ttbar_path = value

# Accessing the file from the online database
tree = uproot.open(background_ttbar_path + ":analysis")

Again, 
    not all weights are important to our analysis. 
In our case, 
    these are:
- `mcWeight` - specific Monte-Carlo weight associated with each event
- `scaleFactor_PILEUP` - scale factor for pileup reweighting
- `scaleFactor_ELE` - scale factor for electron efficiency
- `scaleFactor_MUON`- scale factor for muon efficiency
- `scaleFactor_LepTRIGGER` - scale factor for lepton triggers (TODO not around for new release)

Scale factors are generally related to estimates of the efficiencies and resolutions of detectors.

In [ ]:
data_x,_ = np.histogram(ak.to_numpy(all_data['Data']['mass']), 
                        bins=bin_edges ) # histogram the data
data_x_errors = np.sqrt( data_x ) # statistical error on the data

signal_x = ak.to_numpy(all_data[r'Signal ($m_H$ = 125 GeV)']['mass']) # histogram the signal
signal_weights = ak.to_numpy(all_data[r'Signal ($m_H$ = 125 GeV)'].totalWeight) # get the weights of the signal events
signal_color = samples[r'Signal ($m_H$ = 125 GeV)']['color'] # get the colour for the signal bar

mc_x = [] # define list to hold the Monte Carlo histogram entries
mc_weights = [] # define list to hold the Monte Carlo weights
mc_colors = [] # define list to hold the colors of the Monte Carlo bars
mc_labels = [] # define list to hold the legend labels of the Monte Carlo bars

for s in samples: # loop over samples
    if s not in ['Data', r'Signal ($m_H$ = 125 GeV)']: # if not data nor signal
        mc_x.append( ak.to_numpy(all_data[s]['mass']) ) # append to the list of Monte Carlo histogram entries
        mc_weights.append( ak.to_numpy(all_data[s].totalWeight) ) # append to the list of Monte Carlo weights
        mc_colors.append( samples[s]['color'] ) # append to the list of Monte Carlo bar colors
        mc_labels.append( s ) # append to the list of Monte Carlo legend labels

# *************
# Main plot 
# *************
fig, main_axes = plt.subplots(figsize=(12, 8))

# plot the data points
main_axes.errorbar(x=bin_centres, y=data_x, yerr=data_x_errors,
                    fmt='ko', # 'k' means black and 'o' is for circles 
                    label='Data') 

# plot the Monte Carlo bars
mc_heights = main_axes.hist(mc_x, bins=bin_edges, 
                            weights=mc_weights, stacked=True, 
                            color=mc_colors, label=mc_labels )

mc_x_tot = mc_heights[0][-1] # stacked background MC y-axis value

# calculate MC statistical uncertainty: sqrt(sum w^2)
mc_x_err = np.sqrt(np.histogram(np.hstack(mc_x), bins=bin_edges, weights=np.hstack(mc_weights)**2)[0])

# plot the signal bar
signal_heights = main_axes.hist(signal_x, bins=bin_edges, bottom=mc_x_tot, 
                weights=signal_weights, color=signal_color,
                label=r'Signal ($m_H$ = 125 GeV)')

# plot the statistical uncertainty
main_axes.bar(bin_centres, # x
                2*mc_x_err, # heights
                alpha=0.5, # half transparency
                bottom=mc_x_tot-mc_x_err, color='none', 
                hatch="////", width=step_size, label='Stat. Unc.' )

# set the x-limit of the main axes
main_axes.set_xlim( left=xmin, right=xmax ) 

# separation of x axis minor ticks
main_axes.xaxis.set_minor_locator( AutoMinorLocator() ) 

# set the axis tick parameters for the main axes
main_axes.tick_params(which='both', # ticks on both x and y axes
                        direction='in', # Put ticks inside and outside the axes
                        top=True, # draw ticks on the top axis
                        right=True ) # draw ticks on right axis

# x-axis label
main_axes.set_xlabel(r'4-lepton invariant mass $\mathrm{m_{4l}}$ [GeV]',
                    fontsize=13, x=1, horizontalalignment='right' )

# write y-axis label for main axes
main_axes.set_ylabel('Events / '+str(step_size)+' GeV',
                        y=1, horizontalalignment='right') 

# set y-axis limits for main axes
main_axes.set_ylim( bottom=0, top=np.amax(data_x)*2.0 )

# add minor ticks on y-axis for main axes
main_axes.yaxis.set_minor_locator( AutoMinorLocator() ) 

# Add text 'ATLAS Open Data' on plot
plt.text(0.1, # x
            0.93, # y
            'ATLAS Open Data', # text
            transform=main_axes.transAxes, # coordinate system used is that of main_axes
            fontsize=16 ) 

# Add text 'for education' on plot
plt.text(0.1, # x
            0.88, # y
            'for education', # text
            transform=main_axes.transAxes, # coordinate system used is that of main_axes
            style='italic',
            fontsize=12 ) 

# Add energy and luminosity
lumi_used = str(lumi*fraction) # luminosity to write on the plot
plt.text(0.1, # x
            0.82, # y
            '$\sqrt{s}$=13 TeV,$\int$L dt = '+lumi_used+' fb$^{-1}$', # text
            transform=main_axes.transAxes,fontsize=16 ) # coordinate system used is that of main_axes

# Add a label for the analysis carried out
plt.text(0.1, # x
            0.76, # y
            r'$H \rightarrow ZZ^* \rightarrow 4\ell$', # text 
            transform=main_axes.transAxes,fontsize=16 ) # coordinate system used is that of main_axes

# draw the legend
main_axes.legend( frameon=False, fontsize=16 ) # no box around the legend

### Signal Significance

We can do some analysis to study how significant the signal is compared to the background. 
One method is to check a quantity known as the signal significance $S$,
    which is defined by 
$$ S = \frac{N_\text{sig}}{\sqrt{N_\text{bg}}}  $$
where $ N_\text{sig} $ and $N_\text{bg}$ are the number of signal and background points respectively.
A larger $S$ represents a better signal-to-background ratio,
    and a more significant signal peak.
To calculate $N_\text{sig}$, 
    we can look at the plot and sum over the number of events of our Monte-Carlo signal.
The signal range roughly corresponds to the bins from $115 \,\text{GeV}$ to $130 \, \text{GeV}$.
$N_\text{bg}$ then corresponds to the number of background events in those same bins.

In [ ]:
# Signal stacked height
signal_tot = signal_heights[0] + mc_x_tot

# Peak of signal
print(signal_tot[18])

# Neighbouring bins
print(signal_tot[17:20])

# Signal and background events
N_sig = signal_tot[17:20].sum()
N_bg = mc_x_tot[17:20].sum()

# Signal significance calculation
signal_significance = N_sig/np.sqrt(N_bg + 0.3 * N_bg**2) # EXPLAIN THE 0.3
print(f"\nResults:\n{N_sig = }\n{N_bg = }\n{signal_significance = }\n")